In [1]:
import os
import speech_recognition as sr
from moviepy.editor import VideoFileClip
import pandas as pd
import json
from pathlib import Path

In [11]:
import os
import pandas as pd
import speech_recognition as sr
from moviepy.editor import VideoFileClip
import time
import tempfile
from pydub import AudioSegment
from pydub.silence import split_on_silence
import whisper
import warnings
warnings.filterwarnings("ignore")

class VideoTranscriptExtractor:
    def __init__(self, folder_path):
        """
        Initialize the transcript extractor with multiple fallback options
        
        Args:
            folder_path (str): Path to folder containing videos
        """
        self.folder_path = folder_path
        self.recognizer = sr.Recognizer()
        self.supported_formats = ['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv']
        
        # Configure recognizer for better performance
        self.recognizer.energy_threshold = 300
        self.recognizer.dynamic_energy_threshold = True
        self.recognizer.pause_threshold = 0.8
        self.recognizer.operation_timeout = None
        self.recognizer.phrase_threshold = 0.3
        self.recognizer.non_speaking_duration = 0.5
        
        # Initialize Whisper model (optional - will load on first use)
        self.whisper_model = None
        
        # Request delay to avoid rate limiting
        self.request_delay = 0.5
        
    def load_whisper_model(self):
        """Load Whisper model for offline transcription"""
        try:
            if self.whisper_model is None:
                print("Loading Whisper model for offline transcription...")
                self.whisper_model = whisper.load_model("base")
                print("Whisper model loaded successfully!")
            return True
        except Exception as e:
            print(f"Warning: Could not load Whisper model: {e}")
            return False
        
    def extract_id_emotion(self, filename):
        """
        Extract ID and emotion from filename like '1_anger.mp4'
        
        Args:
            filename (str): Video filename
            
        Returns:
            tuple: (id, emotion)
        """
        name_without_ext = os.path.splitext(filename)[0]
        parts = name_without_ext.split('_', 1)
        
        if len(parts) == 2:
            return parts[0], parts[1]
        else:
            return filename, "unknown"
    
    def get_video_duration(self, video_path):
        """
        Get video duration in seconds
        
        Args:
            video_path (str): Path to video file
            
        Returns:
            float: Duration in seconds
        """
        try:
            video = VideoFileClip(video_path)
            duration = video.duration
            video.close()
            return duration
        except Exception as e:
            print(f"Error getting duration for {video_path}: {str(e)}")
            return 0
    
    def format_duration(self, seconds):
        """
        Format duration from seconds to MM.SS format
        
        Args:
            seconds (float): Duration in seconds
            
        Returns:
            str: Formatted duration (e.g., "1.40" for 1 minute 40 seconds)
        """
        minutes = int(seconds // 60)
        remaining_seconds = int(seconds % 60)
        return f"{minutes}.{remaining_seconds:02d}"
    
    def video_to_audio(self, video_path, audio_path):
        """
        Convert video to audio file with better audio processing
        
        Args:
            video_path (str): Path to video file
            audio_path (str): Path for output audio file
        """
        try:
            video = VideoFileClip(video_path)
            
            # Extract audio with better settings
            audio = video.audio
            if audio is None:
                print(f"No audio track found in {video_path}")
                video.close()
                return False
                
            # Write audio file with optimal settings for speech recognition
            audio.write_audiofile(
                audio_path, 
                verbose=False, 
                logger=None,
                codec='pcm_s16le',  # Better codec for speech recognition
                ffmpeg_params=['-ar', '16000', '-ac', '1']  # 16kHz mono
            )
            
            video.close()
            return True
        except Exception as e:
            print(f"Error converting {video_path} to audio: {str(e)}")
            return False
    
    def preprocess_audio(self, audio_path):
        """
        Preprocess audio to improve transcription quality
        
        Args:
            audio_path (str): Path to audio file
            
        Returns:
            str: Path to processed audio file
        """
        try:
            # Load audio
            audio = AudioSegment.from_wav(audio_path)
            
            # Normalize audio
            audio = audio.normalize()
            
            # Apply noise reduction (simple approach)
            if len(audio) > 1000:  # Only if audio is longer than 1 second
                # Split on silence to remove empty parts
                chunks = split_on_silence(
                    audio, 
                    min_silence_len=500,
                    silence_thresh=audio.dBFS-14,
                    keep_silence=500
                )
                
                # Rejoin chunks
                if chunks:
                    audio = sum(chunks)
            
            # Save processed audio
            processed_path = audio_path.replace('.wav', '_processed.wav')
            audio.export(processed_path, format="wav", parameters=["-ar", "16000", "-ac", "1"])
            
            return processed_path
            
        except Exception as e:
            print(f"Error preprocessing audio: {e}")
            return audio_path
    
    def transcribe_with_whisper(self, audio_path):
        """
        Transcribe audio using Whisper (offline)
        
        Args:
            audio_path (str): Path to audio file
            
        Returns:
            str: Transcribed text
        """
        try:
            if not self.load_whisper_model():
                return None
                
            result = self.whisper_model.transcribe(
                audio_path,
                language=None,  # Auto-detect language
                fp16=False,
                verbose=False
            )
            
            text = result["text"].strip()
            detected_language = result.get("language", "unknown")
            
            if text:
                print(f"Whisper detected language: {detected_language}")
                return text
            
        except Exception as e:
            print(f"Whisper transcription error: {e}")
            
        return None
    
    def detect_language_and_transcribe(self, audio_path):
        """
        Detect language and transcribe audio to text with multiple fallback methods
        
        Args:
            audio_path (str): Path to audio file
            
        Returns:
            str: Transcribed text
        """
        # Preprocess audio for better quality
        processed_audio_path = self.preprocess_audio(audio_path)
        
        try:
            with sr.AudioFile(processed_audio_path) as source:
                # Adjust for ambient noise with longer duration
                self.recognizer.adjust_for_ambient_noise(source, duration=1.5)
                audio_data = self.recognizer.record(source)
            
            # Method 1: Try Google Speech Recognition with multiple languages
            languages = ['id-ID', 'en-US', 'ms-MY', 'zh-CN']  # Added more languages
            
            for lang in languages:
                try:
                    # Add delay to avoid rate limiting
                    time.sleep(self.request_delay)
                    
                    text = self.recognizer.recognize_google(
                        audio_data, 
                        language=lang,
                        show_all=False
                    )
                    
                    if text and text.strip() and len(text.strip()) > 2:
                        # Clean up temporary processed file
                        if processed_audio_path != audio_path:
                            try:
                                os.remove(processed_audio_path)
                            except:
                                pass
                        return text.strip()
                        
                except sr.UnknownValueError:
                    continue
                except sr.RequestError as e:
                    print(f"Google Speech Recognition error for {lang}: {e}")
                    # Increase delay after error
                    time.sleep(2)
                    continue
            
            # Method 2: Try Whisper (offline) if Google fails
            print("Trying Whisper offline transcription...")
            whisper_result = self.transcribe_with_whisper(processed_audio_path)
            if whisper_result:
                # Clean up temporary processed file
                if processed_audio_path != audio_path:
                    try:
                        os.remove(processed_audio_path)
                    except:
                        pass
                return whisper_result
            
            # Method 3: Try offline Sphinx as last resort
            try:
                text = self.recognizer.recognize_sphinx(audio_data)
                if text and text.strip():
                    # Clean up temporary processed file
                    if processed_audio_path != audio_path:
                        try:
                            os.remove(processed_audio_path)
                        except:
                            pass
                    return text.strip()
            except Exception as e:
                print(f"Sphinx recognition error: {e}")
            
            # If all methods fail, return a meaningful message
            return "Audio tidak dapat dikenali - kemungkinan tidak ada suara atau kualitas audio rendah"
                    
        except Exception as e:
            print(f"Error processing audio: {str(e)}")
            return f"Error processing audio: {str(e)}"
        finally:
            # Clean up temporary processed file
            if processed_audio_path != audio_path:
                try:
                    os.remove(processed_audio_path)
                except:
                    pass
    
    def process_videos(self):
        """
        Process all videos in the folder and extract transcripts with duration
        
        Returns:
            list: List of dictionaries containing video info and transcripts
        """
        results = []
        temp_audio_dir = "temp_audio"
        
        # Create temporary directory for audio files
        os.makedirs(temp_audio_dir, exist_ok=True)
        
        # Get all video files
        video_files = [f for f in os.listdir(self.folder_path) 
                      if any(f.lower().endswith(ext) for ext in self.supported_formats)]
        
        # Sort files numerically if possible
        try:
            video_files.sort(key=lambda x: int(os.path.splitext(x)[0].split('_')[0]))
        except:
            video_files.sort()
        
        total_files = len(video_files)
        print(f"Found {total_files} video files to process...")
        print("Using enhanced transcription with multiple fallback methods...")
        
        successful_transcripts = 0
        failed_transcripts = 0
        
        for i, filename in enumerate(video_files, 1):
            print(f"\nProcessing {i}/{total_files}: {filename}")
            
            # Extract ID and emotion
            video_id, emotion = self.extract_id_emotion(filename)
            
            # Paths
            video_path = os.path.join(self.folder_path, filename)
            audio_path = os.path.join(temp_audio_dir, f"{video_id}_{emotion}.wav")
            
            # Get video duration
            duration_seconds = self.get_video_duration(video_path)
            duration_formatted = self.format_duration(duration_seconds)
            
            # Convert video to audio
            if self.video_to_audio(video_path, audio_path):
                # Extract transcript with enhanced language detection
                transcript = self.detect_language_and_transcribe(audio_path)
                
                # Check if transcription was successful
                if not transcript.startswith("Error") and not transcript.startswith("Audio tidak"):
                    successful_transcripts += 1
                    print(f"✓ SUCCESS: {transcript[:60]}...")
                else:
                    failed_transcripts += 1
                    print(f"⚠ FAILED: {transcript}")
                
                # Clean up temporary audio file
                try:
                    os.remove(audio_path)
                except:
                    pass
            else:
                transcript = "Error: Could not extract audio from video"
                failed_transcripts += 1
                print(f"✗ AUDIO EXTRACTION FAILED")
            
            # Store result
            result = {
                'id': video_id,
                'text': transcript,
                'durasi': duration_formatted
            }
            results.append(result)
            
            # Progress update
            success_rate = (successful_transcripts / i) * 100 if i > 0 else 0
            print(f"Progress: {success_rate:.1f}% success rate ({successful_transcripts}/{i} successful)")
        
        # Clean up temporary directory
        try:
            os.rmdir(temp_audio_dir)
        except:
            pass
        
        print(f"\n{'='*60}")
        print(f"FINAL RESULTS:")
        print(f"Total processed: {total_files}")
        print(f"Successful transcriptions: {successful_transcripts}")
        print(f"Failed transcriptions: {failed_transcripts}")
        print(f"Success rate: {(successful_transcripts/total_files)*100:.1f}%")
        print(f"{'='*60}")
            
        return results
    
    def save_to_csv(self, results, output_filename="video_transcripts.csv"):
        """
        Save results to CSV file in the exact format requested
        
        Args:
            results (list): List of transcript results
            output_filename (str): Output CSV filename
        """
        # Create DataFrame with exact column order: id, text, durasi
        df = pd.DataFrame(results, columns=['id', 'text', 'durasi'])
        df.to_csv(output_filename, index=False, encoding='utf-8')
        
        print(f"\nResults saved to: {output_filename}")
        
        # Display preview
        print("\nPreview hasil (5 baris pertama):")
        print(df.head().to_string(index=False, max_colwidth=50))
        
        # Show success statistics
        successful_rows = df[~df['text'].str.startswith(('Error', 'Audio tidak', 'Could not'))].shape[0]
        total_rows = df.shape[0]
        print(f"\nStatistics:")
        print(f"Total videos: {total_rows}")
        print(f"Successfully transcribed: {successful_rows}")
        print(f"Success rate: {(successful_rows/total_rows)*100:.1f}%")


def main():
    # Konfigurasi - GANTI PATH INI SESUAI FOLDER VIDEO ANDA
    FOLDER_PATH = r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\downloaded_videos"  # Ganti dengan path folder video Anda
    OUTPUT_FILENAME = "video_transcripts_enhanced.csv"   # Nama file output
    
    # Pastikan folder exists
    if not os.path.exists(FOLDER_PATH):
        print(f"Error: Folder {FOLDER_PATH} tidak ditemukan!")
        print("Silakan ubah FOLDER_PATH pada kode dengan path yang benar.")
        return
    
    # Initialize extractor
    extractor = VideoTranscriptExtractor(FOLDER_PATH)
    
    # Process videos
    print("Memulai ekstraksi transkrip dan durasi dengan metode enhanced...")
    print("Sistem akan menggunakan multiple fallback methods:")
    print("1. Google Speech Recognition (multiple languages)")
    print("2. Whisper (offline)")
    print("3. Sphinx (offline)")
    print("-" * 60)
    
    results = extractor.process_videos()
    
    # Save results to CSV
    extractor.save_to_csv(results, OUTPUT_FILENAME)
    
    print(f"\n🎉 Selesai! {len(results)} video berhasil diproses.")
    print(f"📄 File output: {OUTPUT_FILENAME}")


# Contoh penggunaan langsung
def process_folder_directly(folder_path, output_file="hasil_transkrip_enhanced.csv"):
    """
    Fungsi untuk memproses folder secara langsung
    
    Args:
        folder_path (str): Path ke folder video
        output_file (str): Nama file output CSV
    """
    extractor = VideoTranscriptExtractor(folder_path)
    results = extractor.process_videos()
    extractor.save_to_csv(results, output_file)
    return results


if __name__ == "__main__":
    main()

Memulai ekstraksi transkrip dan durasi dengan metode enhanced...
Sistem akan menggunakan multiple fallback methods:
1. Google Speech Recognition (multiple languages)
2. Whisper (offline)
3. Sphinx (offline)
------------------------------------------------------------
Found 200 video files to process...
Using enhanced transcription with multiple fallback methods...

Processing 1/200: 1.mp4
Error preprocessing audio: [WinError 2] The system cannot find the file specified
✓ SUCCESS: di sebelah saya sudah ada baik bj40 yang akan kita gunakan b...
Progress: 100.0% success rate (1/1 successful)

Processing 2/200: 2.mp4
Error preprocessing audio: [WinError 2] The system cannot find the file specified
✓ SUCCESS: Civic yang sudah dimodifikasi full carbon jadi meskipun dia ...
Progress: 100.0% success rate (2/2 successful)

Processing 3/200: 3.mp4
Error preprocessing audio: [WinError 2] The system cannot find the file specified
✓ SUCCESS: nama tempat itu gua musang dan kita mampir salat di sebua

In [13]:
import os
import whisper
from moviepy.editor import VideoFileClip

def transkrip_video(video_path, model_size="base"):
    """
    Transkrip video pilihan dengan Whisper (offline, akurat untuk audio panjang)
    
    Args:
        video_path (str): path ke video (.mp4, .avi, .mov, dll)
        model_size (str): ukuran model whisper ("tiny", "base", "small", "medium", "large")
    
    Returns:
        str: hasil transkrip
    """
    # Load model whisper
    print("🔄 Loading Whisper model...")
    model = whisper.load_model(model_size)
    print("✅ Model loaded!")

    # Ekstrak audio sementara
    print("🎬 Extracting audio from video...")
    video = VideoFileClip(video_path)
    duration = video.duration
    if duration > 600:  # 600 detik = 10 menit
        print(f"⚠️ Warning: Video berdurasi {duration:.2f} detik (>10 menit). Hanya akan diproses 10 menit pertama.")
        video = video.subclip(0, 600)  # ambil 10 menit pertama
    
    audio_path = "temp_audio.wav"
    video.audio.write_audiofile(audio_path, verbose=False, logger=None, codec="pcm_s16le", ffmpeg_params=["-ar", "16000", "-ac", "1"])
    video.close()

    # Transkripsi dengan Whisper
    print("📝 Transcribing with Whisper...")
    result = model.transcribe(audio_path, fp16=False, verbose=False)
    
    # Bersihkan file audio sementara
    if os.path.exists(audio_path):
        os.remove(audio_path)

    print("✅ Transkripsi selesai!")
    return result["text"].strip()


# Contoh penggunaan:
if __name__ == "__main__":
    video_file = r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\downloaded_videos\128.mp4"
    teks = transkrip_video(video_file, model_size="medium")  # ganti "base" → "small/medium" untuk akurasi lebih tinggi
    print("\n=== HASIL TRANSKRIP ===\n")
    print(teks)


🔄 Loading Whisper model...


100%|█████████████████████████████████████| 1.42G/1.42G [01:15<00:00, 20.4MiB/s]


✅ Model loaded!
🎬 Extracting audio from video...
⚠️ Warning: Video berdurasi 631.79 detik (>10 menit). Hanya akan diproses 10 menit pertama.
📝 Transcribing with Whisper...


FileNotFoundError: [WinError 2] The system cannot find the file specified